# Row 1: the edge model alone

Runs the decider by itself (no cues, no scene graphs) and scores it. What this tests and why is in
`docs/RATIONALE.md` (section 4, row 1; decisions D5, D9, D12, D13). Run the cells in order from the repo root.

**Outputs worth pushing back to GitHub:** everything under `results/cache/` (small JSONL, a few hundred KB per split).

## 0. Setup and GPU check

In [ ]:
%pip install -q -e ".[models]"
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, f"{p.total_memory/1e9:.1f} GB")

## 1. Fetch the videos (val first: 1.4 GB; dev 2.0 GB; test 1.6 GB)

In [ ]:
!python scripts/ovo_tar_remote.py fetch --samples data/processed/ovo_far_val.jsonl --out data/raw/ovo/src_videos

## 2. Smoke test: two of the shortest val questions

Check this finishes without errors and that the printed tick counts look right before launching a long run.
First run downloads the model (about 7 GB for the 3B model).

In [ ]:
!python scripts/run_decider_cache.py --samples data/processed/ovo_far_val.jsonl --limit 2 --shortest --set edge.model_id=Qwen/Qwen2.5-VL-3B-Instruct

## 2b. If a decision takes more than about a second: find out why

Prints where the model lives, GPU memory use, and splits the time into input preparation and the model forward pass at three frame sizes.

In [ ]:
!python scripts/diagnose_speed.py --set edge.model_id=Qwen/Qwen2.5-VL-3B-Instruct

## 3. How fast is it? (decides whether a full run is practical)

Prints the median time per decision and a projection for all three splits (7,863 decisions in total).

In [ ]:
import glob, json, statistics
rows = [json.loads(l) for f in glob.glob("results/cache/decider/*/*.jsonl") for l in open(f) if l.strip()]
ms = [r["ms"] for r in rows]
print(f"{len(rows)} decisions stored; median {statistics.median(ms):.0f} ms, p90 {sorted(ms)[int(.9*len(ms))]:.0f} ms")
print(f"projected for all 7,863 decisions: {statistics.mean(ms)*7863/3.6e6:.1f} hours of model time")

# sanity: is the model putting its probability on Yes/No at all?
import math
mass = [math.exp(r["lp_yes"]) + math.exp(r["lp_no"]) for r in rows]
print(f"probability mass on Yes/No: median {statistics.median(mass):.2f} (should be close to 1)")

## 4. Full run on the val split (resumable: if it stops, run this cell again)

In [ ]:
!python scripts/run_decider_cache.py --samples data/processed/ovo_far_val.jsonl --set edge.model_id=Qwen/Qwen2.5-VL-3B-Instruct

## 5. Score it: sweep thresholds and cooldowns on val

Pick the best (tau, cooldown) on **val**. Do not tune on test (decision D6). Compare the best score against the
floors in `docs/RATIONALE.md` section 5 (a coin flip scores about 21).

In [ ]:
import glob
cache = sorted(glob.glob("results/cache/decider/Qwen2.5-VL-3B-Instruct__query_only__*"))[-1]
print(cache)
!python scripts/replay_decider.py --cache {cache} --samples data/processed/ovo_far_val.jsonl

## 6. Push the results so they can be read

Run in a terminal:
```
git add results/cache && git commit -m "Row 1 decider cache, val" && git pull --rebase origin abhi && git push origin abhi
```